In [ ]:
# Performance Analytics Notebook
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import linregress
import plotly.express as px

BASE=Path("..")
nav=pd.read_csv(BASE/"data/processed/nav_history_clean.csv")
benchmark=pd.read_csv(BASE/"data/processed/10_benchmark_indices_clean.csv")
funds=pd.read_csv(BASE/"data/processed/01_fund_master_clean.csv")

nav['date']=pd.to_datetime(nav['date'])
benchmark['date']=pd.to_datetime(benchmark['date'])
nav=nav.sort_values(['amfi_code','date'])

CHARTS_DIR=BASE/'reports'/'charts'
CHARTS_DIR.mkdir(parents=True,exist_ok=True)

# Daily Returns
nav['daily_return']=nav.groupby('amfi_code')['nav'].pct_change()

fig=px.histogram(nav.dropna(),x='daily_return',nbins=100,title='Daily Return Distribution')
fig.write_image(CHARTS_DIR/'daily_return_distribution.png',scale=3)
fig.show()

# CAGR
def calc_cagr(df,yrs):
    end=df['date'].max()
    start=end-pd.DateOffset(years=yrs)
    temp=df[df['date']>=start]
    if len(temp)<2:return np.nan
    s=temp.iloc[0]['nav'];e=temp.iloc[-1]['nav']
    return ((e/s)**(1/yrs)-1)

cagr=[]
for code,g in nav.groupby('amfi_code'):
    cagr.append([code,calc_cagr(g,1),calc_cagr(g,3),calc_cagr(g,5)])
cagr=pd.DataFrame(cagr,columns=['amfi_code','cagr_1yr','cagr_3yr','cagr_5yr'])

# Sharpe & Sortino
rf=0.065/252
metrics=[]
for code,g in nav.groupby('amfi_code'):
    r=g['daily_return'].dropna()
    if len(r)<2:continue
    sharpe=((r.mean()-rf)/r.std())*np.sqrt(252)
    down=r[r<0]
    sortino=((r.mean()-rf)/down.std())*np.sqrt(252) if len(down)>1 else np.nan
    running=g['nav'].cummax()
    dd=(g['nav']/running)-1
    maxdd=dd.min()
    metrics.append([code,sharpe,sortino,maxdd])
metrics=pd.DataFrame(metrics,columns=['amfi_code','sharpe_ratio','sortino_ratio','max_drawdown'])

# Alpha Beta
bench=benchmark[benchmark['index_name'].str.contains('100',case=False,na=False)].copy()
bench=bench.sort_values('date')
bench['bench_return']=bench['close_value'].pct_change()

ab=[]
for code,g in nav.groupby('amfi_code'):
    x=g[['date','daily_return']].merge(bench[['date','bench_return']],on='date').dropna()
    if len(x)<30:continue
    beta,alpha,r,p,se=linregress(x['bench_return'],x['daily_return'])
    ab.append([code,alpha*252,beta])
ab=pd.DataFrame(ab,columns=['amfi_code','alpha','beta'])

result=funds.merge(cagr,on='amfi_code',how='left').merge(metrics,on='amfi_code',how='left').merge(ab,on='amfi_code',how='left')

result['return_rank']=result['cagr_3yr'].rank(pct=True)
result['sharpe_rank']=result['sharpe_ratio'].rank(pct=True)
result['alpha_rank']=result['alpha'].rank(pct=True)
result['expense_rank']=1-result['expense_ratio_pct'].rank(pct=True)
result['dd_rank']=result['max_drawdown'].rank(pct=True)

result['fund_score']=100*(0.30*result['return_rank']+0.25*result['sharpe_rank']+0.20*result['alpha_rank']+0.15*result['expense_rank']+0.10*result['dd_rank'])

result.sort_values('fund_score',ascending=False,inplace=True)

result.to_csv(BASE/'reports'/'fund_scorecard.csv',index=False)
result[['amfi_code','scheme_name','alpha','beta']].to_csv(BASE/'reports'/'alpha_beta.csv',index=False)

top5=result.head(5)
fig=px.bar(top5,x='fund_score',y='scheme_name',orientation='h',title='Top 5 Fund Scorecard')
fig.write_image(CHARTS_DIR/'fund_scorecard.png',scale=3)
fig.show()

print(result[['scheme_name','fund_score']].head())
